In [14]:
import numpy as np
import pandas as pd
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from matplotlib import pyplot as plt
from torchvision.datasets import MNIST
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [15]:
# Define a transform to convert images to tensors and normalize them
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) # Normalize with mean and std dev
])

# Load the training dataset
train_dataset = MNIST(
    root='./data',         # Directory to store the data
    train=True,            # Request the training set
    download=True,         # Download the dataset if not available
    transform=transform    # Apply the defined transform
)

# Load the test dataset
test_dataset = MNIST(
    root='./data',
    train=False,           # Request the test set
    download=True,
    transform=transform
)

# Define DataLoaders to batch and shuffle the data
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True           # Shuffle training data
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False          # No need to shuffle test data
)

In [16]:
train_dataset, test_dataset

(Dataset MNIST
     Number of datapoints: 60000
     Root location: ./data
     Split: Train
     StandardTransform
 Transform: Compose(
                ToTensor()
                Normalize(mean=(0.5,), std=(0.5,))
            ),
 Dataset MNIST
     Number of datapoints: 10000
     Root location: ./data
     Split: Test
     StandardTransform
 Transform: Compose(
                ToTensor()
                Normalize(mean=(0.5,), std=(0.5,))
            ))

In [17]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Entrada: 1 canal (P&B), Saída: 32 filtros, Kernel: 3x3
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        # Entrada: 32, Saída: 64 filtros, Kernel: 3x3
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        # Dropout para evitar overfitting
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        # Camadas Lineares (Total de pixels após o pooling: 64 * 14 * 14)
        self.fc1 = nn.Linear(64 * 14 * 14, 128)
        self.fc2 = nn.Linear(128, 10) # 10 classes para os dígitos 0-9

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        # Reduz a imagem de 28x28 para 14x14
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        # "Achata" a matriz para um vetor
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [18]:
device = "cuda"
model = SimpleCNN().to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [19]:
epochs = 10

model.train() # Coloca o modelo em modo de treinamento

for epoch in range(1, epochs + 1):
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        # Enviar dados para GPU ou CPU
        data, target = data.to(device), target.to(device)

        # 1. Zerar os gradientes do otimizador
        optimizer.zero_grad()
        
        # 2. Forward pass (Previsão)
        output = model(data)
        
        # 3. Calcular a perda (Erro)
        loss = loss_fn(output, target)
        
        # 4. Backward pass (Calcular gradientes)
        loss.backward()
        
        # 5. Atualizar pesos
        optimizer.step()

        running_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f"Época: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] "
                  f"Loss: {loss.item():.4f}")

    print(f"--- Fim da Época {epoch} | Média de Loss: {running_loss/len(train_loader):.4f} ---")

Época: 1 [0/60000] Loss: 2.2825
Época: 1 [6400/60000] Loss: 0.2907
Época: 1 [12800/60000] Loss: 0.2150
Época: 1 [19200/60000] Loss: 0.1405
Época: 1 [25600/60000] Loss: 0.1293
Época: 1 [32000/60000] Loss: 0.1625
Época: 1 [38400/60000] Loss: 0.0619
Época: 1 [44800/60000] Loss: 0.2354
Época: 1 [51200/60000] Loss: 0.1961
Época: 1 [57600/60000] Loss: 0.1026
--- Fim da Época 1 | Média de Loss: 0.2403 ---
Época: 2 [0/60000] Loss: 0.1350
Época: 2 [6400/60000] Loss: 0.1738
Época: 2 [12800/60000] Loss: 0.1174
Época: 2 [19200/60000] Loss: 0.0484
Época: 2 [25600/60000] Loss: 0.0185
Época: 2 [32000/60000] Loss: 0.0272
Época: 2 [38400/60000] Loss: 0.0945
Época: 2 [44800/60000] Loss: 0.0293
Época: 2 [51200/60000] Loss: 0.0701
Época: 2 [57600/60000] Loss: 0.0074
--- Fim da Época 2 | Média de Loss: 0.0918 ---
Época: 3 [0/60000] Loss: 0.2314
Época: 3 [6400/60000] Loss: 0.0400
Época: 3 [12800/60000] Loss: 0.0431
Época: 3 [19200/60000] Loss: 0.1729
Época: 3 [25600/60000] Loss: 0.0958
Época: 3 [32000/60000

In [43]:

image, target = train_dataset[0]


image, target


(tensor([[[-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000, -1.0000,
           -1.0000, -1.0000, -1.000

In [44]:
model.eval()

with torch.no_grad():
    image_tensor = image.unsqueeze(0).to(device)
    output = model(image_tensor)
    prediction = output.argmax(dim=1, keepdim=True)
    pred = prediction.item()

In [45]:
target, pred

(5, 5)